<a href="https://colab.research.google.com/github/Shanmuganathan75/QM640-WALSH-CAPSTONE/blob/main/%5B03G%5D-Kappa_Check.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# QM 640 Capstone — Step 3g: Cohen's Kappa Inter-Rater Reliability Check

Compares your classifications (in `screening_worksheet.csv`, merged via
`03f`) against your independent re-coder's classifications (in
`screening_recode_sample.csv`) on their overlapping subsample.

**Run this after `03f_merge_review.ipynb`**, and only once both you and
your independent re-coder have finished classifying your respective files
and pushed them to the repo. If either file still has blank
`announcement_type` values, this will report "not enough re-coded rows
yet" rather than a real kappa - that's expected, not a bug, until both
passes are actually complete.

## Cell 1 — Clone (or pull) the repo

Run this first, every session. `BASE_DIR` is the single fixed path every
other cell in this notebook reads from and writes to.

In [ ]:
import os
from google.colab import userdata

GITHUB_USERNAME = "Shanmuganathan75"      # edit if different
REPO_NAME = "QM640-WALSH-CAPSTONE"        # must match the repo URL EXACTLY (hyphens included)
GITHUB_TOKEN = userdata.get('GITHUB_TOKEN')  # set once via Colab Secrets (key icon, left sidebar)

BASE_DIR = f"/content/{REPO_NAME}"
remote_url = f"https://{GITHUB_TOKEN}@github.com/{GITHUB_USERNAME}/{REPO_NAME}.git"

if not os.path.exists(os.path.join(BASE_DIR, ".git")):
    # Wipe any stale/incomplete folder from a previous failed attempt before retrying
    !rm -rf {BASE_DIR}
    !git clone {remote_url} {BASE_DIR}
else:
    !git -C {BASE_DIR} pull

# Fail loudly instead of silently continuing with a non-git folder -
# this is exactly the bug that caused "fatal: not a git repository" earlier.
if not os.path.exists(os.path.join(BASE_DIR, ".git")):
    raise RuntimeError(
        f"Clone failed: no .git folder found at {BASE_DIR}.\n"
        f"Check that GITHUB_USERNAME (\'{GITHUB_USERNAME}\') and REPO_NAME "
        f"(\'{REPO_NAME}\') exactly match your repo URL (case and hyphens "
        f"included), and that GITHUB_TOKEN is set in Colab Secrets with "
        f"Contents: Read and write access."
    )

for sub in ["scripts", "data/raw", "data/processed/returns", "results"]:
    os.makedirs(os.path.join(BASE_DIR, sub), exist_ok=True)

!git -C {BASE_DIR} config user.email "your_email@example.com"
!git -C {BASE_DIR} config user.name "Shanmuganathan Ekambaram"

print("Repo ready at:", BASE_DIR)

Cloning into '/content/QM640-WALSH-CAPSTONE'...
remote: Enumerating objects: 282, done.
remote: Counting objects: 100% (54/54), done.
remote: Compressing objects: 100% (44/44), done.
remote: Total 282 (delta 24), reused 30 (delta 10), pack-reused 228 (from 1)
Receiving objects: 100% (282/282), 2.89 MiB | 5.38 MiB/s, done.
Resolving deltas: 100% (121/121), done.
Repo ready at: /content/QM640-WALSH-CAPSTONE


## Cell 2 — Install dependencies

In [ ]:
!pip install -q pandas scikit-learn

## Cell 3 — Configuration

In [ ]:
import os

RAW_DIR = os.path.join(BASE_DIR, "data/raw")
SCREENING_FILE = os.path.join(RAW_DIR, "screening_worksheet.csv")
RECODE_FILE = os.path.join(RAW_DIR, "screening_recode_sample.csv")

## Calculate Cohen's kappa

In [ ]:
import pandas as pd
from sklearn.metrics import cohen_kappa_score


def calculate_kappa():
    original = pd.read_csv(SCREENING_FILE)
    recode = pd.read_csv(RECODE_FILE)

    merged = recode.merge(
        original[["accession_no", "announcement_type"]], on="accession_no", how="left"
    )
    merged = merged.dropna(subset=["announcement_type", "recoder_announcement_type"])
    merged = merged[merged["recoder_announcement_type"] != ""]

    if len(merged) < 2:
        print("Not enough re-coded rows yet. Fill in recoder_announcement_type first.")
        return None

    kappa = cohen_kappa_score(merged["announcement_type"], merged["recoder_announcement_type"])
    print(f"Cohen's kappa (announcement_type, n={len(merged)}): {kappa:.3f}")

    if kappa < 0.70:
        print("\nKAPPA BELOW .70 - per the Synopsis, this triggers a joint review "
              "of the classification criteria before finalizing the full sample.")
        disagreements = merged[merged["announcement_type"] != merged["recoder_announcement_type"]]
        print(disagreements[["company_name", "announcement_type", "recoder_announcement_type"]])
    else:
        print("Kappa meets the .70 threshold - classification is reliable enough to proceed.")
    return kappa


kappa_result = calculate_kappa()

Cohen's kappa (announcement_type, n=127): 1.000
Kappa meets the .70 threshold - classification is reliable enough to proceed.


## Save the kappa result for the Interim/Final Report

In [ ]:
import json

RESULTS_DIR = os.path.join(BASE_DIR, "results")
os.makedirs(RESULTS_DIR, exist_ok=True)

if kappa_result is not None:
    with open(os.path.join(RESULTS_DIR, "kappa_result.json"), "w") as f:
        json.dump({"cohen_kappa": kappa_result}, f, indent=2)
    print(f"Saved -> {RESULTS_DIR}/kappa_result.json")

Saved -> /content/QM640-WALSH-CAPSTONE/results/kappa_result.json


## Commit and push results back to GitHub

In [ ]:
!git -C {BASE_DIR} add "results/kappa_result.json"
!git -C {BASE_DIR} commit -m "Step 3g: Cohen's kappa inter-rater reliability result"
!git -C {BASE_DIR} push

[main 2414055] Step 3g: Cohen's kappa inter-rater reliability result
 1 file changed, 3 insertions(+)
 create mode 100644 results/kappa_result.json
Enumerating objects: 6, done.
Counting objects: 100% (6/6), done.
Delta compression using up to 2 threads
Compressing objects: 100% (3/3), done.
Writing objects: 100% (4/4), 455 bytes | 455.00 KiB/s, done.
Total 4 (delta 1), reused 0 (delta 0), pack-reused 0
remote: Resolving deltas: 100% (1/1), completed with 1 local object.
To https://github.com/Shanmuganathan75/QM640-WALSH-CAPSTONE.git
   c22bde2..2414055  main -> main
